In [ ]:
# Imports

import pandas as pd
import numpy as np
from pathlib import Path
from tqdm.auto import tqdm
import torch
import warnings

warnings.filterwarnings('ignore')

from timesfm3 import TimesFM3Evaluator, ModelConfig

# Configurations

FREQUENCY = 'Daily'
SEASONALITY = 7
HORIZON = 14

TIMESFM_MODEL = "google/timesfm-3.0-pytorch"

CONTEXT_MAX = 512

BATCH_SIZE = 32  

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

data_path = Path('../data/M4')
train_file = data_path / f'{FREQUENCY}-train.csv'
test_file = data_path / f'{FREQUENCY}-test.csv'

print(f"Device: {DEVICE}")

# 2. Evaluation Functions

def smape(y_true, y_pred):
    """Symmetric Mean Absolute Percentage Error"""
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    denominator = (np.abs(y_true) + np.abs(y_pred)) / 2.0
    diff = np.abs(y_true - y_pred) / denominator
    diff[denominator == 0] = 0.0
    return 100 * np.mean(diff)

def mase(y_true, y_pred, y_train, seasonality=1):
    """Mean Absolute Scaled Error"""
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    y_train = np.array(y_train)
    mae = np.mean(np.abs(y_true - y_pred))
    naive_mae = np.mean(np.abs(y_train[seasonality:] - y_train[:-seasonality]))
    if naive_mae == 0:
        return np.nan
    return mae / naive_mae

# 3. Data and Model loading

train_df = pd.read_csv(train_file)
test_df = pd.read_csv(test_file)

print(f"Train: {train_df.shape}")
print(f"Test: {test_df.shape}")
print(f"Series: {len(train_df)}")
print(f"Forecast horizon: {HORIZON} days")

id_to_idx = {sid: i for i, sid in enumerate(train_df.iloc[:, 0])}

tfm3_config = ModelConfig(
    checkpoint_path=TIMESFM_MODEL,
    per_core_batch_size=BATCH_SIZE,
    device=DEVICE,
)
forecaster = TimesFM3Evaluator(tfm3_config)
print(f"Modello caricato: {TIMESFM_MODEL} su {DEVICE}")

# 4. Zero shot inference

series_ids = train_df.iloc[:, 0].tolist()
point_forecasts = {}  

for start in tqdm(range(0, len(series_ids), BATCH_SIZE), desc="TimesFM-3 zero-shot"):
    batch_ids = series_ids[start:start + BATCH_SIZE]
    batch_context = []
    for sid in batch_ids:
        idx = id_to_idx[sid]
        series = train_df.iloc[idx, 1:].dropna().values.astype(np.float32)
        if len(series) > CONTEXT_MAX:
            series = series[-CONTEXT_MAX:]
        batch_context.append(series)

    outputs = list(forecaster.predict_batch(
        batch_context,
        horizon=HORIZON,
        return_quantiles=True,
        use_symmetric_averaging=False,
    ))

    for sid, out in zip(batch_ids, outputs):
        point_forecasts[sid] = np.asarray(out.forecast)

print(f"\nForecast generated for {len(point_forecasts)} series.")

# 5. Evaluation

tfm3_smape_scores = []
tfm3_mase_scores = []

for sid, forecast in tqdm(point_forecasts.items(), desc="Evaluating TimesFM-3"):
    idx = id_to_idx[sid]
    train_series = train_df.iloc[idx, 1:].dropna().values
    test_series = test_df.iloc[idx, 1:].dropna().values

    h = min(len(test_series), len(forecast))
    y_true = test_series[:h]
    y_pred = forecast[:h]

    s = smape(y_true, y_pred)
    m = mase(y_true, y_pred, train_series, seasonality=1)
    if not np.isnan(s):
        tfm3_smape_scores.append(s)
    if not np.isnan(m):
        tfm3_mase_scores.append(m)

print("\n" + "="*80)
print(" TIMESFM-3 (zero-shot) RESULTS")
print("="*80)
print(f"   sMAPE: {np.mean(tfm3_smape_scores):.4f}")
print(f"   MASE:  {np.mean(tfm3_mase_scores):.4f}")

In [ ]:
# --- canonical calib/test split
CONTEXT_LENGTH = 90
MIN_SERIES_LENGTH = CONTEXT_LENGTH + HORIZON + 2  
N_SERIES_CALIB, N_SERIES_TEST = 500, 1000

lengths = train_df.iloc[:, 1:].notna().sum(axis=1)
filtered_ids = train_df.iloc[:, 0][lengths >= MIN_SERIES_LENGTH].tolist()
calib_ids = filtered_ids[:N_SERIES_CALIB]
test_ids = filtered_ids[N_SERIES_CALIB:N_SERIES_CALIB + N_SERIES_TEST]

# --- calib forecast
calib_forecasts = {}
for start in tqdm(range(0, len(calib_ids), BATCH_SIZE), desc="TimesFM-3 calib"):
    batch_ids = calib_ids[start:start + BATCH_SIZE]
    batch_context = []
    for sid in batch_ids:
        idx = id_to_idx[sid]
        series = train_df.iloc[idx, 1:].dropna().values.astype(np.float32)
        series = series[:-HORIZON]           # hold out last 14 points
        series = series[-CONTEXT_LENGTH:]    # contesto fisso, canonico
        batch_context.append(series)

    outputs = list(forecaster.predict_batch(
        batch_context,
        horizon=HORIZON,
        return_quantiles=True,
        use_symmetric_averaging=False,
    ))
    for sid, out in zip(batch_ids, outputs):
        calib_forecasts[sid] = np.asarray(out.forecast)

# --- test forecast
test_forecasts = {}
for start in tqdm(range(0, len(test_ids), BATCH_SIZE), desc="TimesFM-3 test"):
    batch_ids = test_ids[start:start + BATCH_SIZE]
    batch_context = []
    for sid in batch_ids:
        idx = id_to_idx[sid]
        series = train_df.iloc[idx, 1:].dropna().values.astype(np.float32)
        series = series[-CONTEXT_LENGTH:]    # contesto fisso, canonico
        batch_context.append(series)

    outputs = list(forecaster.predict_batch(
        batch_context,
        horizon=HORIZON,
        return_quantiles=True,
        use_symmetric_averaging=False,
    ))
    for sid, out in zip(batch_ids, outputs):
        test_forecasts[sid] = np.asarray(out.forecast)

# --- export ---
RESULTS_DIR = Path("../results")
RESULTS_DIR.mkdir(exist_ok=True)
MODEL_NAME = "TimesFM3"

def export_forecasts(forecasts_dict, ids, model_name, split):
    rows = [{"unique_id": uid, **{f"h{i+1}": v for i, v in enumerate(forecasts_dict[uid])}}
            for uid in ids]
    pd.DataFrame(rows).to_csv(RESULTS_DIR / f"{split}_{model_name}.csv", index=False)

export_forecasts(calib_forecasts, calib_ids, MODEL_NAME, "calib")
export_forecasts(test_forecasts, test_ids, MODEL_NAME, "test")

print("Exported:", MODEL_NAME)